# Example 2: 使用Modular Strategy执行多个Job

In [1]:
import os
import numpy as np

from ABQflow import BatchAbaqusProcessor, JobSpec, PreparationSpec, HookSpec
from ABQflow import generate_from_array, degenerate_from_array

%reload_ext autoreload
%autoreload 2

ABAQUS_CAE = 'C:/Applications/SIMULIA/Commands/abaqus.bat'
CWD = os.getcwd()

In [2]:
param_names = ['youngs_modulus', 'load_magnitude']
param_values = np.array([
	[200000, 2000],
	[210000, 3000],
	[220000, 4000],
	[230000, 5000]
])

base_job_spec = JobSpec(
	job_name = "planar_stress_multiple",
	workflow = "modular",
	preparation = PreparationSpec(
		kind = "inp_based",
		source_path = "./examples/cae_file/planar_stress_template.inp",
	),
	pre_extraction = [
		HookSpec(
			script_path = "./examples/extraction_scripts/get_total_mass.py",
			tasks = [
				{"result_name": "total_mass",},
			]
		)
	],
	post_extraction = [
		HookSpec(
			script_path = "./examples/extraction_scripts/get_max_stress_mises.py",
			tasks = [
				{"result_name": "max_stress_mises",},
				{"result_name": "max_displacement",},
			]
		)
	]
)

spec_list = generate_from_array(
	samples_array = param_values,
	param_names = param_names,
	base_spec  = base_job_spec
)

import pprint

pprint.pprint(spec_list)

[JobSpec(job_name='planar_stress_multiple_0001',
         workflow='modular',
         preparation=PreparationSpec(kind='inp_based',
                                     source_path='./examples/cae_file/planar_stress_template.inp',
                                     params={'load_magnitude': 2000.0,
                                             'youngs_modulus': 200000.0},
                                     options={}),
         preflight=None,
         monolithic_script=None,
         monolithic_params={},
         pre_extraction=[HookSpec(script_path='./examples/extraction_scripts/get_total_mass.py',
                                  tasks=[{'result_name': 'total_mass'}])],
         post_extraction=[HookSpec(script_path='./examples/extraction_scripts/get_max_stress_mises.py',
                                   tasks=[{'result_name': 'max_stress_mises'},
                                          {'result_name': 'max_displacement'}])],
         meta={}),
 JobSpec(job_name='planar_st

In [3]:
processor = BatchAbaqusProcessor(
	batch_data = spec_list,
	base_output_dir = os.path.join(CWD, "examples/02_BatchParameterizedJob/output"),
	cpus_per_job = 12,
	duplicate_mode = "overwrite",
	abaqus_exe = ABAQUS_CAE,
)

In [4]:
outcomes = processor.run_batch(
	num_parallel_jobs = 2,
)

Output()

In [5]:
outcomes

[JobOutcome(job_name='planar_stress_multiple_0001', status='COMPLETED', results={'total_mass': 0.00032066262255247625, 'max_stress_mises': 4525.26025390625, 'max_displacement': 4.189039707183838}, error=None, diagnostics=None),
 JobOutcome(job_name='planar_stress_multiple_0002', status='COMPLETED', results={'total_mass': 0.00032066262255247625, 'max_stress_mises': 6787.890625, 'max_displacement': 5.984342098236084}, error=None, diagnostics=None),
 JobOutcome(job_name='planar_stress_multiple_0003', status='COMPLETED', results={'total_mass': 0.00032066262255247625, 'max_stress_mises': 9050.5205078125, 'max_displacement': 7.616435527801514}, error=None, diagnostics=None),
 JobOutcome(job_name='planar_stress_multiple_0004', status='COMPLETED', results={'total_mass': 0.00032066262255247625, 'max_stress_mises': 11313.1513671875, 'max_displacement': 9.106608390808105}, error=None, diagnostics=None)]

In [6]:
Y = degenerate_from_array(
	outcomes = outcomes,
	output_names = ["total_mass", "max_stress_mises", "max_displacement"],
)
print(f"output: {Y}")

output: [[3.20662623e-04 4.52526025e+03 4.18903971e+00]
 [3.20662623e-04 6.78789062e+03 5.98434210e+00]
 [3.20662623e-04 9.05052051e+03 7.61643553e+00]
 [3.20662623e-04 1.13131514e+04 9.10660839e+00]]
